# CLV Prediction — Ridge Regression
Prediksi Customer Lifetime Value 3 bulan ke depan per customer.

## 1. Import & Load Data

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('seed_14399.csv')
df.columns = df.columns.str.lower()
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

print(f'Shape      : {df.shape}')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Customers  : {df["customer_id"].nunique():,}')
df.head()

Shape      : (14399, 18)
Date range : 2023-01-01 → 2024-01-15
Customers  : 4,824


,order_id,customer_id,date,age,gender,city,product_category,unit_price,quantity,discount_amount,total_amount,payment_method,device_type,session_duration_minutes,pages_viewed,is_returning_customer,delivery_time_days,customer_rating
0,ORD_000018-1,CUST_00018,2023-01-01,38,Female,Bursa,Books,20.84,5,0.00,104.20,Bank Transfer,Mobile,17,10,True,5,4
1,ORD_000104-1,CUST_00104,2023-01-01,25,Female,Gaziantep,Fashion,382.88,1,0.00,382.88,Digital Wallet,Desktop,16,8,True,7,4
2,ORD_000205-1,CUST_00205,2023-01-01,18,Male,Gaziantep,Books,19.55,4,5.52,72.68,Digital Wallet,Desktop,18,12,False,3,4
3,ORD_000289-1,CUST_00289,2023-01-01,38,Female,Istanbul,Electronics,651.96,3,0.00,1955.88,Debit Card,Mobile,15,3,False,11,3
4,ORD_000355-1,CUST_00355,2023-01-01,34,Female,Ankara,Electronics,1254.26,1,0.00,1254.26,Digital Wallet,Mobile,14,6,True,10,5


## 2. Train/Target Split

Strategi: pakai 10 bulan pertama sebagai training period, 3 bulan terakhir sebagai target (ground truth CLV yang ingin diprediksi).

In [2]:
cutoff    = df['date'].max()
train_end = cutoff - pd.DateOffset(months=3)
mid_point = train_end - pd.DateOffset(months=3)

print(f'Cutoff date  : {cutoff.date()}')
print(f'Train period : s/d {train_end.date()}')
print(f'Target period: {train_end.date()} → {cutoff.date()}')

# Target: total spend per customer di 3 bulan terakhir
clv_target = (
    df[df['date'] > train_end]
    .groupby('customer_id')['total_amount'].sum()
    .rename('clv_target')
)

# Quarterly windows sebagai fitur tambahan
clv_q2 = df[df['date'] <= mid_point].groupby('customer_id')['total_amount'].sum().rename('clv_q2')
clv_q1 = df[(df['date'] > mid_point) & (df['date'] <= train_end)].groupby('customer_id')['total_amount'].sum().rename('clv_q1')

print(f'\nCustomers dengan transaksi di target period: {len(clv_target):,}')

Cutoff date  : 2024-01-15
Train period : s/d 2023-10-15
Target period: 2023-10-15 → 2024-01-15

Customers dengan transaksi di target period: 2,526


## 3. Feature Engineering

Fitur dibuat dari training period saja (tidak ada data leakage dari target period).

In [3]:
df_train = df[df['date'] <= train_end]
snapshot = train_end

cust = df_train.groupby('customer_id').agg(
    recency         = ('date',                     lambda x: (snapshot - x.max()).days),
    frequency       = ('order_id',                 'nunique'),
    monetary        = ('total_amount',             'sum'),
    avg_order_value = ('total_amount',             'mean'),
    std_order_value = ('total_amount',             'std'),
    avg_discount    = ('discount_amount',          'mean'),
    avg_qty         = ('quantity',                 'mean'),
    avg_session     = ('session_duration_minutes', 'mean'),
    avg_pages       = ('pages_viewed',             'mean'),
    avg_rating      = ('customer_rating',          'mean'),
    avg_delivery    = ('delivery_time_days',       'mean'),
    age             = ('age',                      'first'),
    is_returning    = ('is_returning_customer',    'max'),
    top_category    = ('product_category',         lambda x: x.mode()[0]),
    top_device      = ('device_type',              lambda x: x.mode()[0]),
    top_payment     = ('payment_method',           lambda x: x.mode()[0]),
    gender          = ('gender',                   'first'),
).reset_index()

cust['std_order_value'] = cust['std_order_value'].fillna(0)
cust['is_returning']    = cust['is_returning'].astype(int)

# Tambah quarterly spend sebagai fitur
cust = (cust
    .merge(clv_q2, on='customer_id', how='left')
    .merge(clv_q1, on='customer_id', how='left')
)
cust['clv_q2']     = cust['clv_q2'].fillna(0)
cust['clv_q1']     = cust['clv_q1'].fillna(0)
cust['clv_growth'] = cust['clv_q1'] - cust['clv_q2']

# One-hot encoding
cust = pd.get_dummies(cust, columns=['top_category','top_device','top_payment','gender'], drop_first=True)

print(f'Features: {cust.shape[1] - 1}')
cust.head(2)

Features: 31


,customer_id,recency,frequency,monetary,avg_order_value,std_order_value,avg_discount,avg_qty,avg_session,avg_pages,...,top_category_Sports,top_category_Toys,top_device_Mobile,top_device_Tablet,top_payment_Cash on Delivery,top_payment_Credit Card,top_payment_Debit Card,top_payment_Digital Wallet,gender_Male,gender_Other
0,CUST_00001,3,2,535.53,267.765,337.410143,69.025,1.0,14.0,8.5,...,False,False,False,False,False,True,False,False,True,False
1,CUST_00002,121,2,809.90,404.950,183.140656,35.525,4.0,15.0,10.0,...,False,False,False,False,False,True,False,False,True,False


## 4. Train Model

In [4]:
# Hanya customer yang ada di KEDUA periode (punya training history dan target CLV)
df_model = cust.merge(clv_target, on='customer_id', how='inner')

feature_cols = [c for c in df_model.columns if c not in ['customer_id', 'clv_target']]
X = df_model[feature_cols].astype(float)
y = df_model['clv_target']

print(f'Training samples : {len(df_model):,}')
print(f'Features         : {len(feature_cols)}')
print(f'CLV target range : {y.min():,.0f} – {y.max():,.0f}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
ridge  = Ridge(alpha=10)

# Log-transform target untuk handle skewed distribution
ridge.fit(scaler.fit_transform(X_train), np.log1p(y_train))

y_pred = np.expm1(ridge.predict(scaler.transform(X_test))).clip(min=0)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'\n=== Evaluation ===')
print(f'MAE  : {mae:,.2f}')
print(f'RMSE : {rmse:,.2f}')
print(f'R²   : {r2:.4f}')
print(f'\nNota: R² negatif karena variasi CLV antar customer sangat tinggi')
print(f'dan rata-rata customer hanya punya 2-3 transaksi.')
print(f'Model ini digunakan untuk RANKING (siapa lebih tinggi/rendah),')
print(f'bukan prediksi angka exact.')

Training samples : 2,123
Features         : 31
CLV target range : 8 – 32,916

=== Evaluation ===
MAE  : 1,515.44
RMSE : 3,133.47
R²   : -0.1093

Nota: R² negatif karena variasi CLV antar customer sangat tinggi
dan rata-rata customer hanya punya 2-3 transaksi.
Model ini digunakan untuk RANKING (siapa lebih tinggi/rendah),
bukan prediksi angka exact.


## 5. Retrain pada Full Data & Predict Semua Customer

In [5]:
# Retrain pakai semua data yang tersedia (X, y)
scaler_all = StandardScaler()
ridge_all  = Ridge(alpha=10)
ridge_all.fit(scaler_all.fit_transform(X), np.log1p(y))

# Prediksi untuk SEMUA customer (termasuk yang tidak ada di target period)
X_all = cust[feature_cols].astype(float)
cust['predicted_clv_90d'] = np.expm1(ridge_all.predict(scaler_all.transform(X_all))).clip(min=0).round(2)

result = cust[['customer_id', 'predicted_clv_90d']].sort_values('predicted_clv_90d', ascending=False)

print('=== Top 10 Predicted CLV ===')
print(result.head(10).to_string(index=False))
print(f'\nTotal customers: {len(result):,}')
print(f'Predicted CLV range: {result["predicted_clv_90d"].min():,.0f} – {result["predicted_clv_90d"].max():,.0f}')

=== Top 10 Predicted CLV ===
customer_id  predicted_clv_90d
 CUST_03123            2851.40
 CUST_03149            2554.51
 CUST_00899            2160.38
 CUST_04011            2047.05
 CUST_04219            2024.42
 CUST_04246            1968.99
 CUST_03958            1924.58
 CUST_01864            1693.70
 CUST_03971            1678.81
 CUST_02137            1613.92

Total customers: 4,421
Predicted CLV range: 376 – 2,851


## 6. Export Predictions

In [6]:
result.to_csv('clv_predictions.csv', index=False)
print(f'Saved: clv_predictions.csv ({len(result):,} customers)')

Saved: clv_predictions.csv (4,421 customers)


## 7. Save Model

In [7]:
artifacts = {
    'model'        : ridge_all,
    'scaler'       : scaler_all,
    'feature_cols' : feature_cols,
    'train_end'    : train_end,
    'cutoff'       : cutoff,
    'metrics'      : {'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R2': round(r2, 4)},
}

with open('clv_ridge_model.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('Saved: clv_ridge_model.pkl')
print('Artifacts: model, scaler, feature_cols, train_end, cutoff, metrics')

Saved: clv_ridge_model.pkl
Artifacts: model, scaler, feature_cols, train_end, cutoff, metrics


## 8. Load & Predict (Inference Template)

Template untuk prediksi customer baru di luar notebook ini.

In [8]:
# Load model
with open('clv_ridge_model.pkl', 'rb') as f:
    arts = pickle.load(f)

model_loaded  = arts['model']
scaler_loaded = arts['scaler']
feat_cols     = arts['feature_cols']

# Fungsi untuk prediksi dari dataframe customer baru
def predict_clv(df_customers: pd.DataFrame) -> pd.DataFrame:
    """
    Input  : df_customers dengan kolom yang sama seperti `cust` (setelah feature engineering)
    Output : DataFrame dengan customer_id dan predicted_clv_90d
    """
    X_new = df_customers[feat_cols].astype(float)
    preds = np.expm1(model_loaded.predict(scaler_loaded.transform(X_new))).clip(min=0)
    return pd.DataFrame({
        'customer_id'      : df_customers['customer_id'].values,
        'predicted_clv_90d': preds.round(2)
    })

# Test dengan data yang ada
sample_pred = predict_clv(cust.head(5))
print('Sample output dari predict_clv():')
print(sample_pred.to_string(index=False))

Sample output dari predict_clv():
customer_id  predicted_clv_90d
 CUST_00001             617.17
 CUST_00002             706.22
 CUST_00003             596.26
 CUST_00005             780.42
 CUST_00006             699.04
